In [1]:
# ============================================================
# NB04b_Stalwarts (FIXED)
# Growth + valuation screen. Correctly applies PEG as
# upper bound (lower PEG = cheaper).
#
# Reads:  silver_lynch_features, stalwart_parameters.csv
# Writes: gold_stalwart_candidates, gold_stalwart_screen
# ============================================================

import uuid, numpy as np, pandas as pd
from datetime import datetime

RUN_ID = str(uuid.uuid4())
SCREEN_DATE = str(datetime.now().date())

# Load config
try:
    cfg = (spark.read.format("csv").option("header", "true")
           .load("Files/config/stalwart_parameters.csv")
           .toPandas().set_index("parameter")["value"].to_dict())
except Exception:
    cfg = {"min_market_cap_cr": 5000, "min_ocf_to_ni": 0.80,
           "max_debt_to_equity": 0.50, "min_interest_coverage": 5.0,
           "min_roe_5y": 0.12, "eps_cagr_min": 0.08, "eps_cagr_max": 0.16,
           "pe_vs_own_median_max": 1.5, "pe_vs_peer_median_max": 1.3,
           "max_dividend_adjusted_peg": 1.5,
           "target_stalwart_count": 25, "max_per_industry": 4}

def f(k, d):
    try: return float(cfg.get(k, d))
    except: return float(d)

# Load features
df = spark.table("silver_lynch_features").toPandas()
if df.empty:
    raise RuntimeError("silver_lynch_features is empty. Run NB03 first.")

# Add audit tracking columns to df FIRST
df["screen_date"] = SCREEN_DATE
df["run_id"] = RUN_ID

# Gates
df["g_mcap"]    = df["market_cap_cr"].notna() & (df["market_cap_cr"] >= f("min_market_cap_cr", 5000))
df["g_ocf_ni"]  = df["ocf_to_ni_4y"].isna() | (df["ocf_to_ni_4y"] >= f("min_ocf_to_ni", 0.80))
df["g_debt"]    = df["debt_to_equity"].isna() | (df["debt_to_equity"] <= f("max_debt_to_equity", 0.50))
df["g_cov"]     = df["interest_coverage"].isna() | (df["interest_coverage"] >= f("min_interest_coverage", 5.0))
df["g_roe"]     = df["min_roe_4y"].notna() & (df["min_roe_4y"] >= f("min_roe_5y", 0.12))
df["g_growth"]  = df["eps_cagr_3y"].notna() & (df["eps_cagr_3y"] >= f("eps_cagr_min", 0.08)) & (df["eps_cagr_3y"] <= f("eps_cagr_max", 0.16))
df["g_pe_med"]  = df["pe_vs_own_median"].isna() | (df["pe_vs_own_median"] <= f("pe_vs_own_median_max", 1.5))
df["g_pe_peer"] = df["pe_vs_peer_median"].isna() | (df["pe_vs_peer_median"] <= f("pe_vs_peer_median_max", 1.3))
df["g_peg"]     = df["dividend_adjusted_peg"].isna() | (df["dividend_adjusted_peg"] <= f("max_dividend_adjusted_peg", 1.5))

stal_gates = ["g_mcap", "g_ocf_ni", "g_debt", "g_cov", "g_roe",
              "g_growth", "g_pe_med", "g_pe_peer", "g_peg"]

df["all_stal_gates_pass"] = df[stal_gates].all(axis=1)
df["final_status"] = df["all_stal_gates_pass"].map(
    {True: "STALWART_BUY", False: "STALWART_WATCH"}
)

def reason(row):
    if row["all_stal_gates_pass"]:
        return "All stalwart gates passed"
    failed = [g for g in stal_gates if not row[g]]
    names = {"g_mcap": "mcap≥5000Cr", "g_ocf_ni": "OCF/NI≥0.80",
             "g_debt": "D/E≤0.50", "g_cov": "interest_cov≥5x",
             "g_roe": "min_ROE≥12%", "g_growth": "8%≤CAGR≤16%",
             "g_pe_med": "P/E≤1.5x_own_median",
             "g_pe_peer": "P/E≤1.3x_peer_median",
             "g_peg": "PEG≤1.5"}
    return "Watch: failed " + ", ".join(names[g] for g in failed)

df["status_reason"] = df.apply(reason, axis=1)

# Ranking: lowest PEG and highest quality wins
df["stalwart_score"] = (
    (2.0 - df["dividend_adjusted_peg"].fillna(1.5).clip(upper=2.0)) * 40
    + df["eps_cagr_3y"].fillna(0) * 100 * 2
    + df["min_roe_4y"].fillna(0) * 100
)

# Save screens (every stock)
screen_cols = ["symbol", "company_name", "industry", "rule_set",
              "market_cap_cr", "eps_cagr_3y", "min_roe_4y",
              "debt_to_equity", "interest_coverage", "ocf_to_ni_4y",
              "pe_current", "pe_vs_own_median", "pe_vs_peer_median",
              "dividend_adjusted_peg", "all_stal_gates_pass",
              "final_status", "status_reason", "screen_date", "run_id"]

spark.createDataFrame(df[screen_cols]).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_stalwart_screen")

# Candidates selection
candidates = df[df["final_status"] == "STALWART_BUY"].copy()

if not candidates.empty:
    candidates = candidates.sort_values("stalwart_score", ascending=False)
    candidates["industry_rank"] = candidates.groupby("industry").cumcount() + 1
    candidates = candidates[candidates["industry_rank"] <= int(f("max_per_industry", 4))].head(int(f("target_stalwart_count", 25)))
    candidates["candidate_rank"] = range(1, len(candidates) + 1)
    candidates["candidate_type"] = "STALWART"

    cand_cols = ["candidate_rank", "candidate_type", "symbol", "company_name",
                 "industry", "eps_cagr_3y", "min_roe_4y", "dividend_adjusted_peg",
                 "pe_current", "pe_vs_own_median",
                 "stalwart_score", "status_reason", "screen_date", "run_id"]
                 
    spark.createDataFrame(candidates[cand_cols]).write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable("gold_stalwart_candidates")
        
    print(f"✅ SUCCESS: Saved gold_stalwart_candidates ({len(candidates)} stocks)")
    display(candidates[["candidate_rank", "symbol", "company_name", "eps_cagr_3y", "dividend_adjusted_peg"]])
else:
    # Empty Delta table fallback
    empty_cand = pd.DataFrame(columns=["candidate_rank", "candidate_type", "symbol", "company_name",
                                       "industry", "eps_cagr_3y", "min_roe_4y", "dividend_adjusted_peg",
                                       "pe_current", "pe_vs_own_median",
                                       "stalwart_score", "status_reason", "screen_date", "run_id"])
    spark.createDataFrame(empty_cand).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("gold_stalwart_candidates")
    print("No stalwarts met criteria.")

StatementMeta(, 5235ae01-8e6a-471c-a0b5-b6100f1458b8, 3, Finished, Available, Finished, False)

✅ SUCCESS: Saved gold_stalwart_candidates (11 stocks)


SynapseWidget(Synapse.DataFrame, 7d5303e8-f9fd-42fe-b28d-b2205d8ecfb9)